In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 │ ⚙️ CONFIG + LOAD DATASET (UIT-ViQuAD 2.0 validation — HuggingFace)
# ─────────────────────────────────────────────────────────────────────────────

import os
import re
import sys
import json
import random
from pathlib import Path
from datetime import datetime
from datasets import load_dataset

# ── CONFIG ────────────────────────────────────────────────────────────────────
MODEL_NAME        = "/home/drnguyenvinh/Exam-Assistant/Bench_mark/Qwen3-VL-4B-Instruct"
DATASET_PATH      = "5CD-AI/Viet-ShareGPT-4o-Text-VQA"
DATASET_N_SAMPLES = 2000    # None = full dataset; set integer (e.g. 200) for quick testing
RANDOM_SEED       = 42
MAX_NEW_TOKENS    = 256     # FIX #4: raised from 128 — fine-tuned model generates full sentences
TEMPERATURE       = 0.0
BATCH_SIZE        = 8
MAX_MODEL_LEN     = 2048
GPU_MEM_UTIL      = 0.80
RUN_NAME = "qwen3_vl_4b_instruct_baseline_viquad2_val"
OUTPUT_DIR = Path(f"./benchmark_outputs/{RUN_NAME}_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

# ── FIX #5: Neutral system prompt — no FPT-specific framing ──────────────────
# Original prompt mentioning "FPT University" causes domain mismatch with
# ViQuAD 2.0 (Vietnamese Wikipedia). Model refuses or misreads neutral contexts.
SYSTEM_PROMPT = (
    "You are a reading comprehension assistant. "
    "Answer the question based solely on the provided context. "
    "If the answer is not found in the context, respond exactly: "
    "'Tôi không tìm thấy thông tin trong tài liệu'."
)

# ── Helpers ───────────────────────────────────────────────────────────────────
def build_user_message(context: str, question: str) -> str:
    return f"Ngữ cảnh: {context}\n\nCâu hỏi: {question}"

def extract_context_question(user_content):
    """Reused by Cell 3 / Cell 4."""
    user_content = str(user_content).strip()
    context_match = re.search(
        r"(?:Context|Ngữ cảnh)\s*:\s*(.*?)\s*(?:Question|Câu hỏi)\s*:",
        user_content, flags=re.DOTALL | re.IGNORECASE
    )
    question_match = re.search(
        r"(?:Question|Câu hỏi)\s*:\s*(.*)",
        user_content, flags=re.DOTALL | re.IGNORECASE
    )
    context  = context_match.group(1).strip() if context_match else ""
    question = question_match.group(1).strip() if question_match else ""
    return context, question

# ── Load raw dataset ──────────────────────────────────────────────────────────
print(f"Downloading {DATASET_PATH} from HuggingFace...")
try:
    raw_dataset = load_dataset(DATASET_PATH, split="validation")
    print(f"✅ Loaded: {len(raw_dataset)} samples (validation split)")
    print(f"   Columns: {raw_dataset.column_names}")
except Exception as e:
    print(f"❌ Failed to load dataset: {e}")
    print("💡 Tip: Check your internet connection and try again.")
    sys.exit(1)

# ── Preprocess ────────────────────────────────────────────────────────────────
REFUSAL_GOLD = "tôi không tìm thấy thông tin trong tài liệu"

def parse_answers_field(ans_field):
    """
    FIX #1: HuggingFace loads Sequence({'text':..., 'answer_start':...})
    as a dict-of-lists: {"text": ["ans1", "ans2"], "answer_start": [12, 45]}.
    Extract the text list robustly regardless of whether it arrives as
    dict-of-lists or (less common) list-of-dicts.
    """
    if isinstance(ans_field, dict):
        # Standard HF Sequence format → dict of lists
        texts = ans_field.get("text", [])
        return [t.strip() for t in texts if str(t).strip()]
    elif isinstance(ans_field, list):
        # Fallback: list of dicts (some older loaders)
        return [a["text"].strip() for a in ans_field if a.get("text", "").strip()]
    return []

def preprocess_viquad(dataset, n_samples=None, seed=42):
    """
    Normalize UIT-ViQuAD2.0 into list[dict] ready for inference.

    Key fixes applied:
      #1 — Correct Sequence field parsing (dict-of-lists from HF loader)
      #2 — Preserve plausible_answers for unanswerable samples (debug aid)
      #3 — Keep full gold_answers list; Cell 4 will take max score across all
      #5 — Neutral system prompt already set above (no FPT domain framing)
    """
    samples = []

    for item in dataset:
        ctx = item["context"].strip()
        q   = item["question"].strip()
        if not ctx or not q:
            continue

        is_impossible = item.get("is_impossible", False)

        if is_impossible:
            gold_answers      = [REFUSAL_GOLD]
            # FIX #2: capture plausible_answers so Cell 4 can log them
            plausible_answers = parse_answers_field(item.get("plausible_answers") or {})
        else:
            gold_answers = parse_answers_field(item.get("answers") or {})
            if not gold_answers:
                continue    # answerable but no label — skip
            plausible_answers = []

        user_msg = build_user_message(ctx, q)

        samples.append({
            "id":                  item["id"],
            "context":             ctx,
            "question":            q,
            "gold_answers":        gold_answers,       # full list — used by Cell 4 for max-F1
            "gold_answer":         gold_answers[0],    # primary reference (first annotator)
            "plausible_answers":   plausible_answers,  # non-empty only for unanswerable rows
            "is_impossible":       is_impossible,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_msg}
            ]
        })

    if n_samples and len(samples) > n_samples:
        random.seed(seed)
        samples = random.sample(samples, n_samples)

    answerable_count   = sum(1 for s in samples if not s["is_impossible"])
    unanswerable_count = len(samples) - answerable_count
    print(f"✅ Total samples: {len(samples)} "
          f"(answerable: {answerable_count}, unanswerable: {unanswerable_count})")
    return samples


samples = preprocess_viquad(raw_dataset, n_samples=DATASET_N_SAMPLES, seed=RANDOM_SEED)

# ── Preview ───────────────────────────────────────────────────────────────────
print("\n📋 Preview — first 3 samples:")
for i, s in enumerate(samples[:3]):
    label = "UNANSWERABLE" if s["is_impossible"] else "ANSWERABLE"
    print(f"\n[{i+1}] [{label}] ID: {s['id']}")
    print(f"    Question          : {s['question']}")
    print(f"    Gold answers      : {s['gold_answers']}")
    if s["plausible_answers"]:
        print(f"    Plausible answers : {s['plausible_answers']}")
    print(f"    Context (100c)    : {s['context'][:100]}...")

/home/rtx5070tiadmin/FPT_Exam_Product_Support/Exam_AssistAI/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OUTPUT_DIR: /home/rtx5070tiadmin/FPT_Exam_Product_Support/Exam_AssistAI/benchmark_outputs/qwen3_vl_4b_instruct_baseline_viquad2_val_20260617_174101
✅ Loaded: 3814 samples (validation split)
   Columns: ['id', 'uit_id', 'title', 'context', 'question', 'answers', 'is_impossible', 'plausible_answers']
✅ Total samples: 3814 (answerable: 2653, unanswerable: 1161)

📋 Preview — first 3 samples:

[1] [ANSWERABLE] ID: 0001-0001-0001
    Question          : Paris đạt được thành quả gì sau khoảng 4 thế kỷ tính từ ngày Cách mạng Pháp diễn ra?
    Gold answers      : ['trở thành một trong những trung tâm văn hóa của thế giới, thủ đô của nghệ thuật và giải trí']
    Context (100c)    : Paris nằm ở điểm gặp nhau của các hành trình thương mại đường bộ và đường sông, và là trung tâm của ...

[2] [UNANSWERABLE] ID: 0001-0001-0002
    Question          : Vị trí địa lý của Pháp có gì đặc biệt?
    Gold answers      : ['tôi không tìm thấy thông tin trong tài liệu']
    Plausible answers : ['nằm ở điểm gặp 

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 │ 🚀 LOAD QWEN3-VL-4B-INSTRUCT WITH vLLM
# ─────────────────────────────────────────────────────────────────────────────
import io
import sys
import os
import subprocess
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

def _patch_jupyter_streams():
    _null_fd = os.open(os.devnull, os.O_WRONLY)

    class _FilenoPatchedStream:
        def __init__(self, original, null_fd):
            object.__setattr__(self, "_orig", original)
            object.__setattr__(self, "_null_fd", null_fd)

        def fileno(self):
            return object.__getattribute__(self, "_null_fd")

        def write(self, s):
            return object.__getattribute__(self, "_orig").write(s)

        def flush(self):
            return object.__getattribute__(self, "_orig").flush()

        def isatty(self):
            return False

        def readable(self):
            return False

        def writable(self):
            return True

        def __getattr__(self, name):
            return getattr(object.__getattribute__(self, "_orig"), name)

    for stream_name in ("stdout", "stderr"):
        stream = getattr(sys, stream_name)
        try:
            stream.fileno()
        except io.UnsupportedOperation:
            setattr(sys, stream_name, _FilenoPatchedStream(stream, _null_fd))

_patch_jupyter_streams()

print(subprocess.getoutput("nvidia-smi"))

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

llm = LLM(
    model=MODEL_NAME,
    trust_remote_code=True,
    dtype="half",
    quantization=None,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEM_UTIL,
    enforce_eager=True,
    max_num_seqs=BATCH_SIZE,
    limit_mm_per_prompt={"image": 0}
)

sampling_params = SamplingParams(
    temperature=TEMPERATURE,
    max_tokens=MAX_NEW_TOKENS
)

print("✅ Model loaded successfully.")

Wed Jun 17 17:41:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5070 Ti     Off |   00000000:02:00.0  On |                  N/A |
|  0%   44C    P8             37W /  300W |    2294MiB /  16303MiB |     18%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

[transformers] `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.
[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


(EngineCore pid=539880) INFO 06-17 17:41:11 [core.py:112] Initializing a V1 LLM engine (v0.22.0) with config: model='/home/rtx5070tiadmin/FPT_Exam_Product_Support/Exam_AssistAI/Qwen3-VL-4B-Instruct', speculative_config=None, tokenizer='/home/rtx5070tiadmin/FPT_Exam_Product_Support/Exam_AssistAI/Qwen3-VL-4B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_i

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:01<00:01,  1.10s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:02<00:00,  1.40s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:02<00:00,  1.36s/it]
(EngineCore pid=539880) 


(EngineCore pid=539880) INFO 06-17 17:41:15 [default_loader.py:397] Loading weights took 2.76 seconds
(EngineCore pid=539880) INFO 06-17 17:41:15 [gpu_model_runner.py:5132] Model loading took 8.69 GiB memory and 2.995668 seconds
(EngineCore pid=539880) INFO 06-17 17:41:16 [gpu_model_runner.py:6136] Encoder cache will be initialized with a budget of 12288 tokens, and profiled with 1 video items of the maximum feature size.
(EngineCore pid=539880) INFO 06-17 17:41:20 [gpu_worker.py:466] Available KV cache memory: 2.07 GiB
(EngineCore pid=539880) INFO 06-17 17:41:20 [kv_cache_utils.py:1733] GPU KV cache size: 15,088 tokens
(EngineCore pid=539880) INFO 06-17 17:41:20 [kv_cache_utils.py:1734] Maximum concurrency for 2,048 tokens per request: 7.37x


(EngineCore pid=539880) 2026-06-17 17:41:20,992 - INFO - autotuner.py:615 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=539880) 2026-06-17 17:41:21,006 - INFO - autotuner.py:634 - flashinfer.jit: [Autotuner]: Autotuning process ends


(EngineCore pid=539880) INFO 06-17 17:41:21 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(EngineCore pid=539880) INFO 06-17 17:41:21 [core.py:309] init engine (profile, create kv cache, warmup model) took 5.75 s
(EngineCore pid=539880) WARNING 06-17 17:41:21 [vllm.py:1033] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=539880) (EngineCore pid=539880) WARNING 06-17 17:41:21 [vllm.py:1058] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-17 17:41:21 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
(EngineCore pid=539880) INFO 06-17 17:41:21 [vllm.py:1234] Cudagraph is disabled under eager mode
✅ Model loaded suc

(EngineCore pid=539880) WARNING 06-17 17:41:22 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _compute_slot_mapping_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(EngineCore pid=539880) WARNING 06-17 17:49:28 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _triton_mrope_forward. This causes a latency spike; consider extending warmup to cover this shape/config.


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 │ 🔮 INFERENCE
# ─────────────────────────────────────────────────────────────────────────────
import time
from tqdm import tqdm

prompts = []

for s in samples:
    prompt = tokenizer.apply_chat_template(
        s["messages"],
        tokenize=False,
        add_generation_prompt=True
    )
    prompts.append(prompt)

print("Total prompts:", len(prompts))
print("Prompt preview:")
print(prompts[0][:1000])

start_time = time.time()

outputs = llm.generate(
    prompts,
    sampling_params,
    use_tqdm=True
)

total_time = time.time() - start_time

predictions = [
    output.outputs[0].text.strip()
    for output in outputs
]

for s, pred in zip(samples, predictions):
    s["prediction"] = pred

print("✅ Inference completed")
print("Total time:", round(total_time, 2), "seconds")
print("Avg latency:", round(total_time / len(samples), 4), "sec/sample")
print("Throughput:", round(len(samples) / total_time, 2), "samples/sec")

print("\nPrediction example:")
print(predictions[0])

Total prompts: 3814
Prompt preview:
<|im_start|>system
You are a reading comprehension assistant. Answer the question based solely on the provided context. If the answer is not found in the context, respond exactly: 'Tôi không tìm thấy thông tin trong tài liệu'.<|im_end|>
<|im_start|>user
Ngữ cảnh: Paris nằm ở điểm gặp nhau của các hành trình thương mại đường bộ và đường sông, và là trung tâm của một vùng nông nghiệp giàu có. Vào thế kỷ 10, Paris đã là một trong những thành phố chính của Pháp cùng các cung điện hoàng gia, các tu viện và nhà thờ. Từ thế kỷ 12, Paris trở thành một trong những trung tâm của châu Âu về giáo dục và nghệ thuật. Thế kỷ 14, Paris là thành phố quan trọng bậc nhất của Cơ Đốc giáo và trong các thế kỷ 16, 17, đây là nơi diễn ra Cách mạng Pháp cùng nhiều sự kiện lịch sử quan trọng của Pháp và châu Âu. Đến thế kỷ 19 và 20, thành phố trở thành một trong những trung tâm văn hóa của thế giới, thủ đô của nghệ thuật và giải trí.

Câu hỏi: Paris đạt được thành quả gì sau 

Processed prompts: 100%|██████████| 3814/3814 [08:05<00:00,  7.85it/s, est. speed input: 2448.71 toks/s, output: 556.60 toks/s]

✅ Inference completed
Total time: 487.09 seconds
Avg latency: 0.1277 sec/sample
Throughput: 7.83 samples/sec

Prediction example:
Sau khoảng 4 thế kỷ tính từ ngày Cách mạng Pháp diễn ra (tức từ thế kỷ 16 đến thế kỷ 20), Paris trở thành một trong những trung tâm văn hóa của thế giới, thủ đô của nghệ thuật và giải trí.


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 │ 📊 EVALUATION METRICS — ANSWERABLE / UNANSWERABLE + FAITHFULNESS
#             + CONTEXT RECALL
# ─────────────────────────────────────────────────────────────────────────────

import re
import json
import unicodedata
import numpy as np
import pandas as pd
from collections import Counter

REFUSAL_GOLD = "tôi không tìm thấy thông tin trong tài liệu"

REFUSAL_PATTERNS = [
    "tôi không tìm thấy thông tin",
    "không tìm thấy thông tin",
    "không có thông tin",
    "không được cung cấp",
    "không có trong ngữ cảnh",
    "không có trong tài liệu",
    "không thể trả lời",
    "không đủ thông tin"
]

# ── Text utilities ────────────────────────────────────────────────────────────

def normalize_text(text):
    text = str(text).lower().strip()
    text = unicodedata.normalize("NFC", text)
    text = re.sub(
        r"[^\w\sàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ]",
        " ", text
    )
    text = re.sub(r"\s+", " ", text).strip()
    return text

def is_gold_unanswerable(gold):
    return REFUSAL_GOLD in normalize_text(gold)

def is_pred_refusal(pred):
    pred_norm = normalize_text(pred)
    return any(p in pred_norm for p in REFUSAL_PATTERNS)

# ── Standard QA metrics ───────────────────────────────────────────────────────

def compute_em(pred, gold):
    return int(normalize_text(pred) == normalize_text(gold))

def compute_f1(pred, gold):
    pred_tokens = normalize_text(pred).split()
    gold_tokens = normalize_text(gold).split()
    if not pred_tokens and not gold_tokens:
        return 1.0
    if not pred_tokens or not gold_tokens:
        return 0.0
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall    = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

def lcs_length(x, y):
    dp = [[0] * (len(y) + 1) for _ in range(len(x) + 1)]
    for i in range(1, len(x) + 1):
        for j in range(1, len(y) + 1):
            if x[i - 1] == y[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[-1][-1]

def compute_rouge_l(pred, gold):
    pred_tokens = normalize_text(pred).split()
    gold_tokens = normalize_text(gold).split()
    if not pred_tokens or not gold_tokens:
        return 0.0
    lcs = lcs_length(pred_tokens, gold_tokens)
    precision = lcs / len(pred_tokens)
    recall    = lcs / len(gold_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def compute_containment(pred, gold):
    pred_norm = normalize_text(pred)
    gold_norm = normalize_text(gold)
    if not gold_norm:
        return 0
    return int(gold_norm in pred_norm)

# ── Faithfulness ─────────────────────────────────────────────────────────────
# Định nghĩa: prediction phải "có căn cứ" trong context.
# Approximation không dùng LLM judge:
#   - Token-level precision của prediction so với context.
#   - faithfulness = |tokens(pred) ∩ tokens(context)| / |tokens(pred)|
#   - Đối với unanswerable: nếu model từ chối đúng → 1.0 (không hallucinate),
#     nếu model hallucinate → đo bình thường (thường rất thấp).

def compute_faithfulness(pred: str, context: str) -> float:
    """
    Token-level precision: tỷ lệ token trong prediction xuất hiện trong context.
    Cao → prediction bám sát context (faithful).
    Thấp → prediction chứa nhiều nội dung không có trong context (hallucination).
    """
    pred_tokens    = normalize_text(pred).split()
    context_tokens = set(normalize_text(context).split())
    if not pred_tokens:
        return 1.0   # empty pred không thêm thông tin sai
    matched = sum(1 for t in pred_tokens if t in context_tokens)
    return matched / len(pred_tokens)

# ── Context Recall ────────────────────────────────────────────────────────────
# Định nghĩa: gold answer phải được "bao phủ" bởi context.
# Approximation: token-level recall của gold so với context.
#   context_recall = |tokens(gold) ∩ tokens(context)| / |tokens(gold)|
# Nếu gold_unanswerable → context_recall = 1.0
#   (context đúng khi KHÔNG chứa câu trả lời, và mô hình nên từ chối)

def compute_context_recall(gold: str, context: str, gold_unanswerable: bool) -> float:
    """
    Đo mức độ context hỗ trợ gold answer.
    Với unanswerable: luôn = 1.0 vì context đúng là thiếu thông tin.
    """
    if gold_unanswerable:
        return 1.0
    gold_tokens    = normalize_text(gold).split()
    context_tokens = set(normalize_text(context).split())
    if not gold_tokens:
        return 1.0
    matched = sum(1 for t in gold_tokens if t in context_tokens)
    return matched / len(gold_tokens)

# ── Main evaluation loop ──────────────────────────────────────────────────────

# ── Main evaluation loop (FIX #3: max score across all gold_answers) ─────────

rows = []

for s in samples:
    pred             = s["prediction"]
    gold             = s["gold_answer"]         # primary reference
    gold_answers_all = s.get("gold_answers", [gold])  # full list
    context          = s.get("context", "")
    is_impossible    = s.get("is_impossible", False)

    gold_unanswerable = is_impossible
    pred_refusal      = is_pred_refusal(pred)
    answerable        = not gold_unanswerable

    faithfulness   = compute_faithfulness(pred, context)
    context_recall = compute_context_recall(gold, context, gold_unanswerable)

    if answerable:
        # FIX #3: take max score across all reference answers
        em          = max(compute_em(pred, g)      for g in gold_answers_all)
        f1          = max(compute_f1(pred, g)      for g in gold_answers_all)
        rouge_l     = max(compute_rouge_l(pred, g) for g in gold_answers_all)
        containment = max(compute_containment(pred, g) for g in gold_answers_all)

        false_refusal   = int(pred_refusal)
        correct_refusal = 0
        hallucination   = 0

    else:
        em      = int(pred_refusal)
        f1      = 1.0 if pred_refusal else 0.0
        rouge_l = 1.0 if pred_refusal else 0.0
        containment = 0

        false_refusal   = 0
        correct_refusal = int(pred_refusal)
        hallucination   = int(not pred_refusal)

    rows.append({
        "id":          s["id"],
        "question":    s["question"],
        "context":     context,
        "gold_answer": gold,
        "gold_answers_all": json.dumps(gold_answers_all, ensure_ascii=False),
        "prediction":  pred,

        "is_answerable":   int(answerable),
        "is_unanswerable": int(not answerable),
        "pred_refusal":    int(pred_refusal),

        "em":          em,
        "f1":          f1,
        "rouge_l":     rouge_l,
        "containment": containment,

        "faithfulness":   faithfulness,
        "context_recall": context_recall,

        "false_refusal":   false_refusal,
        "correct_refusal": correct_refusal,
        "hallucination":   hallucination,

        "gold_len_words": len(gold.split()),
        "pred_len_words": len(pred.split())
    })

# ... phần summary và display giữ nguyên từ Cell 4 trước
df = pd.DataFrame(rows)

answerable_df   = df[df["is_answerable"]   == 1]
unanswerable_df = df[df["is_unanswerable"] == 1]

summary = {
    "model":   MODEL_NAME,
    "dataset": DATASET_PATH,
    "num_samples": len(df),

    "answerable_samples":   len(answerable_df),
    "unanswerable_samples": len(unanswerable_df),

    "overall_em":      df["em"].mean()      * 100,
    "overall_f1":      df["f1"].mean()      * 100,
    "overall_rouge_l": df["rouge_l"].mean() * 100,

    "answerable_em":           answerable_df["em"].mean()          * 100,
    "answerable_f1":           answerable_df["f1"].mean()          * 100,
    "answerable_rouge_l":      answerable_df["rouge_l"].mean()     * 100,
    "answerable_containment":  answerable_df["containment"].mean() * 100,
    "false_refusal_rate_answerable": answerable_df["false_refusal"].mean() * 100,

    "correct_refusal_rate_unanswerable": unanswerable_df["correct_refusal"].mean() * 100,
    "hallucination_rate_unanswerable":   unanswerable_df["hallucination"].mean()   * 100,

    # ── New ───────────────────────────────────────────────────────────────
    "overall_faithfulness":          df["faithfulness"].mean()   * 100,
    "answerable_faithfulness":       answerable_df["faithfulness"].mean()   * 100,
    "unanswerable_faithfulness":     unanswerable_df["faithfulness"].mean() * 100,

    "overall_context_recall":        df["context_recall"].mean()  * 100,
    "answerable_context_recall":     answerable_df["context_recall"].mean()  * 100,
    # unanswerable context_recall luôn = 100% theo định nghĩa, vẫn log để minh bạch
    "unanswerable_context_recall":   unanswerable_df["context_recall"].mean() * 100,

    "avg_gold_len_words": df["gold_len_words"].mean(),
    "avg_pred_len_words": df["pred_len_words"].mean(),

    "total_inference_time_sec": total_time,
    "avg_latency_sec":          total_time / max(len(df), 1),
    "throughput_samples_per_sec": len(df) / total_time if total_time > 0 else 0
}

summary_df = pd.DataFrame([summary])

print("===== BENCHMARK SUMMARY — SQUAD2 / VIQUAD2 STYLE =====")
display(summary_df)

print("\n===== KEY RESULTS =====")
print(f"Samples: {len(df)}")
print(f"Answerable:   {len(answerable_df)}")
print(f"Unanswerable: {len(unanswerable_df)}")
print()
print(f"Answerable EM:         {summary['answerable_em']:.2f}%")
print(f"Answerable F1:         {summary['answerable_f1']:.2f}%")
print(f"Answerable ROUGE-L:    {summary['answerable_rouge_l']:.2f}%")
print(f"Containment Accuracy:  {summary['answerable_containment']:.2f}%")
print(f"False Refusal Rate:    {summary['false_refusal_rate_answerable']:.2f}%")
print()
print(f"Correct Refusal Rate:  {summary['correct_refusal_rate_unanswerable']:.2f}%")
print(f"Hallucination Rate:    {summary['hallucination_rate_unanswerable']:.2f}%")
print()
print(f"Overall Faithfulness:  {summary['overall_faithfulness']:.2f}%")
print(f"Answerable Faithfulness:   {summary['answerable_faithfulness']:.2f}%")
print(f"Unanswerable Faithfulness: {summary['unanswerable_faithfulness']:.2f}%")
print()
print(f"Overall Context Recall:    {summary['overall_context_recall']:.2f}%")
print(f"Answerable Context Recall: {summary['answerable_context_recall']:.2f}%")
print()
print(f"Overall EM:     {summary['overall_em']:.2f}%")
print(f"Overall F1:     {summary['overall_f1']:.2f}%")

print("\n===== HALLUCINATION EXAMPLES =====")
display(
    df[df["hallucination"] == 1][
        ["id", "question", "gold_answer", "prediction"]
    ].head(10)
)

print("\n===== FALSE REFUSAL EXAMPLES =====")
display(
    df[df["false_refusal"] == 1][
        ["id", "question", "gold_answer", "prediction"]
    ].head(10)
)

print("\n===== LOW FAITHFULNESS EXAMPLES (answerable, faithfulness < 0.3) =====")
display(
    answerable_df[answerable_df["faithfulness"] < 0.3][
        ["id", "question", "context", "gold_answer", "prediction", "faithfulness"]
    ].head(10)
)

===== BENCHMARK SUMMARY — SQUAD2 / VIQUAD2 STYLE =====


,model,dataset,num_samples,answerable_samples,unanswerable_samples,overall_em,overall_f1,overall_rouge_l,answerable_em,answerable_f1,...,answerable_faithfulness,unanswerable_faithfulness,overall_context_recall,answerable_context_recall,unanswerable_context_recall,avg_gold_len_words,avg_pred_len_words,total_inference_time_sec,avg_latency_sec,throughput_samples_per_sec
0,/home/rtx5070tiadmin/FPT_Exam_Product_Support/...,taidng/UIT-ViQuAD2.0,3814,2653,1161,10.959622,38.690136,38.040323,0.414625,40.280504,...,84.237423,63.25905,99.967624,99.953456,100.0,9.561615,49.408233,487.093026,0.127712,7.830126



===== KEY RESULTS =====
Samples: 3814
Answerable:   2653
Unanswerable: 1161

Answerable EM:         0.41%
Answerable F1:         40.28%
Answerable ROUGE-L:    39.35%
Containment Accuracy:  76.93%
False Refusal Rate:    1.85%

Correct Refusal Rate:  35.06%
Hallucination Rate:    64.94%

Overall Faithfulness:  77.85%
Answerable Faithfulness:   84.24%
Unanswerable Faithfulness: 63.26%

Overall Context Recall:    99.97%
Answerable Context Recall: 99.95%

Overall EM:     10.96%
Overall F1:     38.69%

===== HALLUCINATION EXAMPLES =====


,id,question,gold_answer,prediction
1,0001-0001-0002,Vị trí địa lý của Pháp có gì đặc biệt?,tôi không tìm thấy thông tin trong tài liệu,"Từ ngữ cảnh đã nêu, vị trí địa lý của Pháp có ..."
10,0001-0002-0004,Điều gì đã khiến toàn cầu trở thành nơi trung ...,tôi không tìm thấy thông tin trong tài liệu,Điều khiến toàn cầu trở thành nơi trung chuyển...
18,0001-0003-0004,Tên gọi Pháp bắt nguồn từ sự việc gì?,tôi không tìm thấy thông tin trong tài liệu,"Tên gọi Pháp ""Ville lumière"" (Thành phố ánh sá..."
22,0001-0004-0003,Thành phố nào được gọi là thành phố của Hai tì...,tôi không tìm thấy thông tin trong tài liệu,"Thành phố được gọi là ""Thành phố của tình yêu""..."
24,0001-0005-0002,Nguồn gốc tên gọi Mỹ bắt nguồn từ đâu?,tôi không tìm thấy thông tin trong tài liệu,"Nguồn gốc tên gọi ""Mỹ"" trong ngữ cảnh này khôn..."
26,0001-0006-0001,Con sông nào nằm phía nam Paris?,tôi không tìm thấy thông tin trong tài liệu,Con sông nằm phía nam Paris là **sông Seine**.
30,0001-0006-0005,Hai đảo nằm giữa Seine trên dòng sông Paris có...,tôi không tìm thấy thông tin trong tài liệu,Hai đảo nằm giữa Seine trên dòng sông Paris có...
32,0001-0006-0007,Công trình nào nằm dưới đỉnh Montmartre?,tôi không tìm thấy thông tin trong tài liệu,Công trình nằm dưới đỉnh Montmartre là **nhà t...
38,0001-0007-0006,Rừng Paris nằm ở đâu của Vincennes?,tôi không tìm thấy thông tin trong tài liệu,"Rừng Vincennes nằm ở phía Đông của Paris, cụ t..."
42,0001-0008-0004,Sự kiện nào đánh dấu sự hình thành của lưu vực...,tôi không tìm thấy thông tin trong tài liệu,Sự kiện đánh dấu sự hình thành của lưu vực sôn...



===== FALSE REFUSAL EXAMPLES =====


,id,question,gold_answer,prediction
84,0001-0014-0004,Ai đã kêu gọi người dân Paris không bỏ xứ tron...,Thánh Geneviève,Tôi không tìm thấy thông tin trong tài liệu.
126,0001-0022-0003,Versailles dùng để làm gì sau khi nhà vua bị đ...,Quốc hội lập hiến,"Sau khi nhà vua bị đuổi khỏi Versailles, địa đ..."
145,0001-0026-0003,Tàu điện ngầm đầu tiên của Pháp nối liền những...,"Grand Palais, Petit Palais và cầu Alexandre-III",Tài liệu không cung cấp thông tin cụ thể về nh...
166,0001-0030-0003,Những mặt hàng nào đem lại lợi nhuận lớn nhất ...,"sản xuất hàng tiêu dùng, xây dựng, xe hơi, năn...","Dựa trên ngữ cảnh đã cho, các lĩnh vực công ng..."
175,0001-0031-0006,Sự chênh lệch thu nhập giữa nam và nữ ở Quận 8...,"6%, trong khi ở các tỉnh lên đến 10 %",Sự chênh lệch thu nhập giữa nam và nữ ở Quận 8...
220,0001-0040-0005,Tỉ lệ dân di cư đến Pháp cho đến năm 2011 là b...,"23,1%","Theo tài liệu, tỉ lệ dân di cư đến Pháp trong ..."
310,0001-0060-0001,Trung tâm Pompidou được thiết kế theo phong cá...,Kiến trúc đương đại,"Trong ngữ cảnh đã cho, **Trung tâm Pompidou** ..."
316,0001-0060-0007,Các công trình trên các quốc lộ ở Paris được x...,"Guimard, Plumet hay Lavirotte với phong cách T...",Tôi không tìm thấy thông tin trong tài liệu.
407,0001-0079-0001,"Nước nào có nền nghệ thuật,kiến truc,âm nhạc p...",nước Pháp,Tôi không tìm thấy thông tin trong tài liệu.
509,0002-0006-0008,Trong mỗi thị tộc sẽ có bao nhiêu bộ lạc?,341 bộ lạc,"Từ ngữ cảnh, không có thông tin cụ thể cho biế..."



===== LOW FAITHFULNESS EXAMPLES (answerable, faithfulness < 0.3) =====


,id,question,context,gold_answer,prediction,faithfulness
84,0001-0014-0004,Ai đã kêu gọi người dân Paris không bỏ xứ tron...,"Sau thất bại của những người Gaulois, nơi đây ...",Thánh Geneviève,Tôi không tìm thấy thông tin trong tài liệu.,0.222222
316,0001-0060-0007,Các công trình trên các quốc lộ ở Paris được x...,"Trong thế kỷ 20, rất nhiều các kiến trúc sư da...","Guimard, Plumet hay Lavirotte với phong cách T...",Tôi không tìm thấy thông tin trong tài liệu.,0.111111
407,0001-0079-0001,"Nước nào có nền nghệ thuật,kiến truc,âm nhạc p...","Từ thế kỷ 18, nước Pháp có thể xem như đã đi đ...",nước Pháp,Tôi không tìm thấy thông tin trong tài liệu.,0.111111
685,0003-0008-0006,"Tại sao trước khi người châu Ân đến, số người ...",Vào thời điểm người châu Âu thiết lập các khu ...,phải chịu tổn thất do các dịch bệnh truyền nhi...,Tôi không tìm thấy thông tin trong tài liệu.,0.222222
817,0003-0028-0002,Ngành nào là ông vua không ngai trong nền kinh...,"Kể từ đầu thế kỷ XX, sự phát triển của các ngà...",ngành công nghiệp dịch vụ,Tôi không tìm thấy thông tin trong tài liệu.,0.111111
828,0003-0030-0003,Tổ chức nào do Pierre Trudeau tạo ra còn tồn t...,Sự hội nhập của kinh tế Canada với Hoa Kỳ tăng...,Cơ quan Đánh giá đầu tư nước ngoài (FIRA),Tôi không tìm thấy thông tin trong tài liệu.,0.222222
936,0003-0048-0003,Tỉnh nào có số người sử dụng tiếng anh đứng th...,Hiến chương Pháp ngữ 1977 xác định tiếng Pháp ...,Ontario,Tôi không tìm thấy thông tin trong tài liệu.,0.000000
937,0003-0048-0004,Tỉnh nào có số người sử dụng tiếng anh nhiều n...,Hiến chương Pháp ngữ 1977 xác định tiếng Pháp ...,Québec,Tôi không tìm thấy thông tin trong tài liệu.,0.000000
939,0003-0049-0001,Canada có tất cả các loại ngôn ngữ của dân bản...,Các tỉnh khác không có ngôn ngữ chính thức như...,"11 nhóm ngôn ngữ Thổ dân, bao gồm hơn 65 phươn...",Tôi không tìm thấy thông tin trong tài liệu.,0.222222
1073,0004-0012-0006,Cuộc hẹn bất ngờ giữa Saddam Hussein và April ...,"Cuối tháng 7 năm 1990, khi những cuộc thương l...",vùng biên giới với Kuwait,Tôi không tìm thấy thông tin trong tài liệu.,0.222222


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 │ 💾 SAVE RESULTS + CHARTS — QWEN FIXED BENCHMARK
# ─────────────────────────────────────────────────────────────────────────────
import json
import matplotlib.pyplot as plt

results_jsonl = OUTPUT_DIR / "qwen_predictions_fixed_metrics.jsonl"
results_csv = OUTPUT_DIR / "qwen_predictions_fixed_metrics.csv"
summary_csv = OUTPUT_DIR / "qwen_summary_fixed_metrics.csv"
summary_json = OUTPUT_DIR / "qwen_summary_fixed_metrics.json"

with open(results_jsonl, "w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

df.to_csv(results_csv, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")

with open(summary_json, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved fixed metrics:")
print(results_jsonl)
print(results_csv)
print(summary_csv)
print(summary_json)

# Chart 1: Answerable QA metrics
metric_names = ["EM", "Token F1", "ROUGE-L", "Containment", "False Refusal"]
metric_values = [
    summary["answerable_em"],
    summary["answerable_f1"],
    summary["answerable_rouge_l"],
    summary["answerable_containment"],
    summary["false_refusal_rate_answerable"]
]

plt.figure(figsize=(10, 5))
plt.bar(metric_names, metric_values)
plt.ylabel("Percent (%)")
plt.title("Qwen3-VL-4B — Answerable QA Metrics")
plt.ylim(0, 100)
plt.grid(axis="y", alpha=0.3)

for i, v in enumerate(metric_values):
    plt.text(i, v + 1, f"{v:.2f}%", ha="center")

chart_path = OUTPUT_DIR / "qwen_answerable_metrics.png"
plt.savefig(chart_path, dpi=200, bbox_inches="tight")
plt.show()

# Chart 2: Unanswerable behavior
metric_names = ["Correct Refusal", "Hallucination"]
metric_values = [
    summary["correct_refusal_rate_unanswerable"],
    summary["hallucination_rate_unanswerable"]
]

plt.figure(figsize=(7, 5))
plt.bar(metric_names, metric_values)
plt.ylabel("Percent (%)")
plt.title("Qwen3-VL-4B — Unanswerable QA Behavior")
plt.ylim(0, 100)
plt.grid(axis="y", alpha=0.3)

for i, v in enumerate(metric_values):
    plt.text(i, v + 1, f"{v:.2f}%", ha="center")

chart_path = OUTPUT_DIR / "qwen_unanswerable_behavior.png"
plt.savefig(chart_path, dpi=200, bbox_inches="tight")
plt.show()

# Chart 3: Dataset split
split_names = ["Answerable", "Unanswerable"]
split_values = [
    summary["answerable_samples"],
    summary["unanswerable_samples"]
]

plt.figure(figsize=(7, 5))
plt.bar(split_names, split_values)
plt.ylabel("Number of samples")
plt.title("Dataset Split")
plt.grid(axis="y", alpha=0.3)

for i, v in enumerate(split_values):
    plt.text(i, v + 10, str(v), ha="center")

chart_path = OUTPUT_DIR / "dataset_split.png"
plt.savefig(chart_path, dpi=200, bbox_inches="tight")
plt.show()

# Chart 4: F1 distribution on answerable only
plt.figure(figsize=(9, 5))
plt.hist(answerable_df["f1"], bins=20)
plt.xlabel("Token F1")
plt.ylabel("Number of samples")
plt.title("Qwen3-VL-4B — F1 Distribution on Answerable Samples")
plt.grid(axis="y", alpha=0.3)

chart_path = OUTPUT_DIR / "qwen_answerable_f1_distribution.png"
plt.savefig(chart_path, dpi=200, bbox_inches="tight")
plt.show()

print("\n✅ Qwen fixed benchmark results saved to:")
print(OUTPUT_DIR.resolve())

FileNotFoundError: [Errno 2] No such file or directory: 'benchmark_outputs/qwen3_vl_4b_instruct_baseline_viquad2_val_20260617_174101/qwen_predictions_fixed_metrics.jsonl'

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 │ 📝 SAVE QWEN FIXED BENCHMARK REPORT
# ─────────────────────────────────────────────────────────────────────────────

report_path = OUTPUT_DIR / "qwen_benchmark_report_fixed_metrics.md"

report = f"""# Benchmark Report — Qwen3-VL-4B-Instruct Baseline

## Model

`{MODEL_NAME}`

## Dataset

`{DATASET_PATH}`

Number of samples: **{len(df)}**

- Answerable samples: **{summary["answerable_samples"]}**
- Unanswerable samples: **{summary["unanswerable_samples"]}**

## Main Results

### Answerable QA

| Metric | Value |
|---|---:|
| Answerable EM | {summary["answerable_em"]:.2f}% |
| Answerable Token F1 | {summary["answerable_f1"]:.2f}% |
| Answerable ROUGE-L | {summary["answerable_rouge_l"]:.2f}% |
| Containment Accuracy | {summary["answerable_containment"]:.2f}% |
| False Refusal Rate | {summary["false_refusal_rate_answerable"]:.2f}% |

### Unanswerable QA

| Metric | Value |
|---|---:|
| Correct Refusal Rate | {summary["correct_refusal_rate_unanswerable"]:.2f}% |
| Hallucination Rate on Unanswerable | {summary["hallucination_rate_unanswerable"]:.2f}% |

### Overall

| Metric | Value |
|---|---:|
| Overall EM | {summary["overall_em"]:.2f}% |
| Overall Token F1 | {summary["overall_f1"]:.2f}% |
| Overall ROUGE-L | {summary["overall_rouge_l"]:.2f}% |
| Average Latency | {summary["avg_latency_sec"]:.4f} sec/sample |
| Throughput | {summary["throughput_samples_per_sec"]:.2f} samples/sec |

## Interpretation

This benchmark follows a SQuAD2.0 / ViQuAD2.0-style evaluation because the dataset contains both answerable and unanswerable questions.

For answerable samples, EM, Token F1, ROUGE-L, and Containment Accuracy measure answer quality. Containment Accuracy is useful when the model generates a complete sentence while the gold answer is a short span.

For unanswerable samples, Correct Refusal Rate measures whether the model correctly refuses to answer when the context does not contain the answer. Hallucination Rate measures cases where the model produces an answer despite the gold label being unanswerable.

## Output Files

- `qwen_predictions_fixed_metrics.jsonl`
- `qwen_predictions_fixed_metrics.csv`
- `qwen_summary_fixed_metrics.csv`
- `qwen_summary_fixed_metrics.json`
- `qwen_answerable_metrics.png`
- `qwen_unanswerable_behavior.png`
- `dataset_split.png`
- `qwen_answerable_f1_distribution.png`
"""

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)

print("✅ Qwen fixed report saved:")
print(report_path.resolve())

✅ Qwen fixed report saved:
/home/rtx5070tiadmin/FPT_Exam_Product_Support/Exam_AssistAI/benchmark_outputs/qwen3_vl_4b_instruct_baseline_viquad2_val_20260617_172241/qwen_benchmark_report_fixed_metrics.md


In [ ]:
print(prompts[0])
print(predictions[0])

<|im_start|>system
You are a reading comprehension assistant. Answer the question based solely on the provided context. If the answer is not found in the context, respond exactly: 'Tôi không tìm thấy thông tin trong tài liệu'.<|im_end|>
<|im_start|>user
Ngữ cảnh: Paris nằm ở điểm gặp nhau của các hành trình thương mại đường bộ và đường sông, và là trung tâm của một vùng nông nghiệp giàu có. Vào thế kỷ 10, Paris đã là một trong những thành phố chính của Pháp cùng các cung điện hoàng gia, các tu viện và nhà thờ. Từ thế kỷ 12, Paris trở thành một trong những trung tâm của châu Âu về giáo dục và nghệ thuật. Thế kỷ 14, Paris là thành phố quan trọng bậc nhất của Cơ Đốc giáo và trong các thế kỷ 16, 17, đây là nơi diễn ra Cách mạng Pháp cùng nhiều sự kiện lịch sử quan trọng của Pháp và châu Âu. Đến thế kỷ 19 và 20, thành phố trở thành một trong những trung tâm văn hóa của thế giới, thủ đô của nghệ thuật và giải trí.

Câu hỏi: Paris đạt được thành quả gì sau khoảng 4 thế kỷ tính từ ngày Cách mạ

In [ ]:
df["gold_len_words"].describe()
df["pred_len_words"].describe()
low_f1 = df.sort_values("f1").head(20)

for _, row in low_f1.iterrows():
    print("="*20)
    print("QUESTION:", row["question"])
    print("GOLD    :", row["gold_answer"])
    print("PRED    :", row["prediction"])
    print("F1      :", row["f1"])

QUESTION: Giai đoạn quan trọng nhất khi xử lý nước ngọt là gì?
GOLD    : tôi không tìm thấy thông tin trong tài liệu
PRED    : Giai đoạn quan trọng nhất khi xử lý nước ngọt là **việc khử muối** thông qua các công nghệ như bốc hơi chân không, bốc hơi flash hoặc thẩm thấu ngược. Đây là bước thiết yếu để chuyển nước biển thành nước uống được, nhưng cũng là bước rất tiêu tốn năng lượng và không thực tế trong thời đại thuyền buồm thế kỷ 16–19.
F1      : 0.0
QUESTION: Vì sao Antelme có thể giải thoát cho Mitterrand?
GOLD    : tôi không tìm thấy thông tin trong tài liệu
PRED    : Antelme có thể giải thoát cho Mitterrand vì Mitterrand đã may mắn tìm thấy người bạn và là thành viên trong mạng lưới của ông, Robert Antelme, đang bị sốt Rickettsia. Vì Antelme bị nhiễm bệnh này, ông được ra lệnh ở lại trong trại để ngăn ngừa bệnh lây truyền. Do đó, Mitterrand đã thu xếp để ông "bỏ trốn" và gửi ông về Pháp để điều trị — điều này cho thấy Mitterrand đã can thiệp để giải thoát Antelme khỏi tình huống 

In [ ]:
refusal_gold = 0

for s in samples:
    gold = s["gold_answer"].lower()

    if "tôi không tìm thấy thông tin" in gold:
        refusal_gold += 1

print("Unanswerable samples:", refusal_gold)
print("Answerable samples:", len(samples)-refusal_gold)
print("Refusal ratio:", refusal_gold/len(samples)*100)

Unanswerable samples: 1161
Answerable samples: 2653
Refusal ratio: 30.440482433141057
